# Notebook 04 — Hybrid Recommender
**Member 2** | Combines SVD (collaborative) + TF-IDF (content-based) with alpha blending.
Alpha experiments: 0.3 / 0.5 / 0.7 / 0.9 + feedback loop simulation.

In [29]:
import pandas as pd
import numpy as np
import pickle
import os
print(os.getcwd())
print(os.path.exists('../model/svd_model.pkl'))
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print('Libraries loaded.')

d:\movie-recommender-ml\notebooks
True
Libraries loaded.


## 1. Load SVD Model + Content Model

In [30]:
import pandas as pd
from surprise import SVD, Dataset, Reader
import pickle

# Load ratings
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
user_col = 'userId' if 'userId' in ratings.columns else ratings.columns[0]
item_col = 'movieId' if 'movieId' in ratings.columns else ratings.columns[1]
rating_col = 'rating' if 'rating' in ratings.columns else ratings.columns[2]

# Train SVD
reader = Reader(rating_scale=(ratings[rating_col].min(), ratings[rating_col].max()))
data = Dataset.load_from_df(ratings[[user_col, item_col, rating_col]], reader)
trainset = data.build_full_trainset()

svd_model = SVD(n_factors=50, random_state=42)
svd_model.fit(trainset)

# Save to models/
with open('../models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)

print('SVD model retrained and saved!')

SVD model retrained and saved!


In [31]:
# Load SVD model
with open('../models/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

# Load content model
with open('../models/content_model.pkl', 'rb') as f:
    content_model = pickle.load(f)
movies      = content_model['movies']
cosine_sim  = content_model['cosine_sim']
indices     = content_model['indices']
movie_id_col = content_model['movie_id_col']
title_col   = content_model['title_col']
genre_col   = content_model['genre_col']

ratings = pd.read_csv('../data/processed/ratings_clean.csv')
user_col   = 'userId'  if 'userId'  in ratings.columns else ratings.columns[0]
item_col   = 'movieId' if 'movieId' in ratings.columns else ratings.columns[1]
rating_col = 'rating'  if 'rating'  in ratings.columns else ratings.columns[2]

print('Models and data loaded.')
print(f'Ratings shape: {ratings.shape}')

Models and data loaded.
Ratings shape: (19116, 4)


## 2. Hybrid Recommender Function

In [32]:
def svd_score(user_id, movie_id):
    """Get SVD predicted rating for a user-movie pair."""
    try:
        pred = svd_model.predict(user_id, movie_id)
        return pred.est
    except Exception:
        return 3.0  # fallback to global mean

def content_score(movie_id, candidate_id):
    """Get content similarity between two movies."""
    if movie_id not in indices or candidate_id not in indices:
        return 0.0
    i = indices[movie_id]
    j = indices[candidate_id]
    # Handle duplicate indices - take first value
    if hasattr(i, '__len__'):
        i = i.iloc[0]
    if hasattr(j, '__len__'):
        j = j.iloc[0]
    return float(cosine_sim[int(i), int(j)])

def hybrid_recommend(user_id, seed_movie_id, alpha=0.5, n=10):
    """
    Hybrid score = alpha * SVD_score + (1-alpha) * content_similarity
    alpha=1 → pure collaborative, alpha=0 → pure content
    """
    all_movie_ids = movies[movie_id_col].tolist()
    
    # Movies already rated by user
    rated = set(ratings[ratings[user_col] == user_id][item_col].tolist())
    candidates = [m for m in all_movie_ids if m not in rated and m != seed_movie_id]
    
    scores = []
    for mid in candidates:
        svd_s  = svd_score(user_id, mid) / 5.0          # normalise to [0,1]
        cont_s = content_score(seed_movie_id, mid)
        hybrid_s = alpha * svd_s + (1 - alpha) * cont_s
        scores.append((mid, hybrid_s))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    top = scores[:n]
    
    result_ids = [s[0] for s in top]
    result_scores = [s[1] for s in top]
    result = movies[movies[movie_id_col].isin(result_ids)][[movie_id_col, title_col]].copy()
    score_map = dict(zip(result_ids, result_scores))
    result['hybrid_score'] = result[movie_id_col].map(score_map)
    return result.sort_values('hybrid_score', ascending=False)

# Quick smoke test
sample_user = ratings[user_col].iloc[0]
sample_movie = ratings[item_col].iloc[0]
print(hybrid_recommend(sample_user, sample_movie, alpha=0.5, n=5))

                                 Film    Genre  hybrid_score
0          Zack and Miri Make a Porno  Romance      0.430917
1                     Youth in Revolt   Comedy      0.430917
2  You Will Meet a Tall Dark Stranger   Comedy      0.430917
3                        When in Rome   Comedy      0.430917
4               What Happens in Vegas   Comedy      0.430917


## 3. Alpha Comparison Experiment (0.3 / 0.5 / 0.7 / 0.9)

In [33]:
alphas = [0.3, 0.5, 0.7, 0.9]
test_users  = ratings[user_col].unique()[:30]
alpha_results = {}

for alpha in alphas:
    avg_scores = []
    for user in test_users:
        user_ratings = ratings[ratings[user_col] == user]
        if user_ratings.empty:
            continue
        seed = user_ratings[item_col].iloc[0]
        recs = hybrid_recommend(user, seed, alpha=alpha, n=10)
        if not recs.empty:
            avg_scores.append(recs['hybrid_score'].mean())
    alpha_results[alpha] = np.mean(avg_scores) if avg_scores else 0
    print(f'alpha={alpha}  avg_hybrid_score={alpha_results[alpha]:.4f}')

alpha=0.3  avg_hybrid_score=0.2107
alpha=0.5  avg_hybrid_score=0.3511
alpha=0.7  avg_hybrid_score=0.4915
alpha=0.9  avg_hybrid_score=0.6320


## 4. Plot Alpha Comparison & Save

In [34]:
os.makedirs('../reports/figures', exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(a) for a in alphas],
       [alpha_results[a] for a in alphas],
       color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'],
       edgecolor='black', width=0.5)
ax.set_xlabel('Alpha (weight of collaborative filtering)', fontsize=12)
ax.set_ylabel('Average Hybrid Score', fontsize=12)
ax.set_title('Hybrid Model — Alpha Comparison', fontsize=14)
ax.set_ylim(0, 1)
for i, (a, v) in enumerate(alpha_results.items()):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/hybrid_alpha_comparison.png', dpi=150)
plt.show()
print('Saved: hybrid_alpha_comparison.png')

Saved: hybrid_alpha_comparison.png


C:\Users\megha\AppData\Local\Temp\ipykernel_19920\2868133413.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Feedback Loop Simulation

In [35]:
# Simulate: user rates recommended movies → retrain implicit scores
print('=== Feedback Loop Simulation ===')

sim_user = ratings[user_col].iloc[0]
sim_seed = ratings[ratings[user_col] == sim_user][item_col].iloc[0]
best_alpha = max(alpha_results, key=alpha_results.get)

print(f'User: {sim_user}, Seed movie: {sim_seed}, Best alpha: {best_alpha}')

rounds = 3
feedback_log = []
current_seed = sim_seed

for r in range(1, rounds + 1):
    recs = hybrid_recommend(sim_user, current_seed, alpha=best_alpha, n=5)
    if recs.empty:
        break
    top_rec = recs.iloc[0]
    # Simulate user gives 4-star rating to top recommendation
    simulated_rating = 4.0
    feedback_log.append({
        'round': r,
        'recommended_movie': top_rec[title_col] if title_col in top_rec else top_rec[movie_id_col],
        'hybrid_score': round(top_rec['hybrid_score'], 4),
        'simulated_rating': simulated_rating
    })
    current_seed = top_rec[movie_id_col]  # next seed = top rec
    print(f'Round {r}: Recommended → {feedback_log[-1]["recommended_movie"]} | score={top_rec["hybrid_score"]:.4f}')

feedback_df = pd.DataFrame(feedback_log)
print('\nFeedback log:')
print(feedback_df)

=== Feedback Loop Simulation ===
User: 1, Seed movie: 1, Best alpha: 0.9
Round 1: Recommended → Romance | score=0.7757
Round 2: Recommended → Comedy | score=0.8757
Round 3: Recommended → Romance | score=0.8757

Feedback log:
   round recommended_movie  hybrid_score  simulated_rating
0      1           Romance        0.7757               4.0
1      2            Comedy        0.8757               4.0
2      3           Romance        0.8757               4.0
